In [ ]:
# %% [markdown]
# # 07 – Final Paper Visualizations (comparative across sources)

# %%
import plotly.graph_objects as go
from shared_functions import load_metrics, get_results_path

# Load synthetic results
asr_syn = load_metrics("asr_analysis", data_source="synthetic")["asr_values"]
asr_def_syn = load_metrics("sanitization", data_source="synthetic")["asr_defended"]

# Load hotpot results if available
try:
    asr_hot = load_metrics("asr_analysis", data_source="hotpot")["asr_values"]
    asr_def_hot = load_metrics("sanitization", data_source="hotpot")["asr_defended"]
except FileNotFoundError:
    print("Hotpot results not found; skipping.")
    asr_hot = None
    asr_def_hot = None

turn_numbers = list(range(1, len(asr_syn)+1))

# %%
fig = go.Figure()
fig.add_trace(go.Scatter(x=turn_numbers, y=asr_syn, mode='lines+markers', name='Synthetic - No Defense'))
fig.add_trace(go.Scatter(x=turn_numbers, y=asr_def_syn, mode='lines+markers', name='Synthetic - Defended'))
if asr_hot is not None:
    fig.add_trace(go.Scatter(x=turn_numbers, y=asr_hot, mode='lines+markers', name='Hotpot - No Defense'))
    fig.add_trace(go.Scatter(x=turn_numbers, y=asr_def_hot, mode='lines+markers', name='Hotpot - Defended'))
fig.update_layout(title="ASR Decay – Comparison Across Data Sources",
                  xaxis_title="Turn", yaxis_title="ASR")
fig.show()
fig.write_html(get_results_path("figures") / "asr_comparison.html")

# %%
# UMAP projection for synthetic data (or both)
from memorypoison_audit.core.memory_store import MemoryStore
from memorypoison_audit.benchmarks.visualize import Visualizer
import numpy as np

store = MemoryStore()
embeddings = store.get_all_embeddings("asr_notebook")  # from synthetic run
if len(embeddings) > 0:
    labels = ['benign'] * (len(embeddings)-3) + ['malicious'] * 3
    fig2 = Visualizer.plot_umap_embeddings(embeddings, labels, title="UMAP Projection (Synthetic)")
    fig2.write_html(get_results_path("figures") / "umap_synthetic.html")
    fig2.show()